In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc, f1_score, hamming_loss
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from scipy.signal import stft

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents)
                            if (p / "scripts" / "project.py").exists()) / "scripts"))
from project import *  # ROOT, CACHE_FILE, LEAD_SET, MODELS_DIR, RESULTS_DIR, load_cache, ...
MODELS_DIR, RESULTS_DIR = output_dirs()
X_train, y_train, X_test, y_test, class_names, X_feat_train, X_feat_test = load_cache()
tf.keras.utils.set_random_seed(SEED)  # reproducible weights init / shuffling

Lead set  -> lead1 (channels [0])
Signals   -> Train: (19230, 1000, 1) | Test: (2158, 1000, 1)
Features  -> Train: (19230, 24) | Test: (2158, 24)
Targets   -> Classes (5): ['CD', 'HYP', 'MI', 'NORM', 'STTC']


In [3]:
def batch_compute_stft(X_signals, fs=100, nperseg=32, noverlap=16):
    """
    Converts (N, 1000, 3) raw time series into (N, 16, 32, 3) STFT spectrogram grids.
    Matches dimensions achievable with hardware-accelerated esp-dsp routines.
    """
    N, _, channels = X_signals.shape
    specs = np.zeros((N, 16, 32, channels), dtype=np.float32)
    
    for i in range(N):
        for c in range(channels):
            _, _, Zxx = stft(X_signals[i, :, c], fs=fs, nperseg=nperseg, noverlap=noverlap)
            # Diagnostic frequency slice (< 25 Hz) and crop first 32 time windows
            mag = np.abs(Zxx[:16, :32])
            log_spec = np.log1p(mag)
            ptp = np.ptp(log_spec)
            specs[i, :, :, c] = (log_spec - np.min(log_spec)) / (ptp + 1e-8) if ptp > 0 else log_spec
            
    return specs

print("Computing STFT spectrogram grids for Train and Test sets...")
X_spec_train = batch_compute_stft(X_train)
X_spec_test = batch_compute_stft(X_test)
print(f"Spectrogram Tensors: Train {X_spec_train.shape} | Test {X_spec_test.shape}")

def build_time_frequency_2d_cnn(input_shape=(16, 32, 3), num_classes=5):
    inputs = layers.Input(shape=input_shape)
    
    x = layers.SeparableConv2D(16, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.SeparableConv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.SeparableConv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="sigmoid")(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name="TimeFreq_2D_CNN")

cnn_2d = build_time_frequency_2d_cnn(input_shape=X_spec_train.shape[1:], num_classes=len(class_names))
cnn_2d.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="macro_auc")]
)

print("\nTraining Time-Frequency 2D-CNN...")
cb = [
    callbacks.EarlyStopping(monitor="val_macro_auc", mode="max", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
]
cnn_2d.fit(
    X_spec_train, y_train,
    validation_split=0.1,
    epochs=25,
    batch_size=64,
    callbacks=cb,
    verbose=1
)

cnn_2d_probs = cnn_2d.predict(X_spec_test, batch_size=128)

Computing STFT spectrogram grids for Train and Test sets...


Spectrogram Tensors: Train (19230, 16, 32, 1) | Test (2158, 16, 32, 1)



Training Time-Frequency 2D-CNN...


Epoch 1/25


  1/271 ━━━━━━━━━━━━━━━━━━━━ 51:01 11s/step - loss: 0.7352 - macro_auc: 0.4742

  4/271 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.7221 - macro_auc: 0.5241  

... (1,910 outputs left out)

 5/17 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step 

11/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step


In [4]:
all_predictions = {}
all_predictions["Time-Freq 2D-CNN (STFT)"] = cnn_2d_probs
print(f"Time-Frequency 2D-CNN Macro AUC: {roc_auc_score(y_test, cnn_2d_probs, average='macro'):.4f}")

Time-Frequency 2D-CNN Macro AUC: 0.7851


In [5]:
import numpy as np

# For 2D-CNN (Notebook 4)
np.savez_compressed(RESULTS_DIR / "preds_2dcnn.npz", cnn2d=cnn_2d_probs)
cnn_2d.save(MODELS_DIR / "timefreq_2d_cnn.keras")